# 01 — EDA tập nhãn hiệu US

Khám phá `processed/clean.csv` (nhánh 04) + LMDB đã dựng (nhánh 05).

Sinh ≥ 4 hình vào `$DATA_ROOT/processed/eda_figures/`.

In [1]:
import os
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/media/zaibachkhoa/AC84-F995/trademark_data'))
PROC = DATA_ROOT / 'processed'
FIG = PROC / 'eda_figures'
FIG.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('figures ->', FIG)

DATA_ROOT = /media/zaibachkhoa/AC84-F995/trademark_data
figures -> /media/zaibachkhoa/AC84-F995/trademark_data/processed/eda_figures


In [2]:
df = pd.read_csv(PROC / 'clean.csv', keep_default_na=False)
print('clean.csv:', df.shape)
print(df.head(3).to_string())
splits = {p.stem: len(pd.read_csv(p, keep_default_na=False))
          for p in sorted(PROC.glob('split_*.csv'))}
print('splits:', splits)
assert len(df) > 0, 'clean.csv rỗng — chạy nhánh 02-04 trước'
df['word_mark'] = df['word_mark'].astype(str)

clean.csv: (4840, 7)
   serial_number                  image_path   word_mark  n_chars        drawing_kind  mark_drawing_cd publication_date
0       50000007  raw/images_us/50000007.png   X 17 V 27        9   design_with_words                3       2026-10-13
1       50000055  raw/images_us/50000055.png  GABY SKILL       10  standard_character                4       2026-10-13
2       50000056  raw/images_us/50000056.png  JETI SKILL       10  standard_character                4       2026-10-13
splits: {'split_test_us': 242, 'split_train': 4356, 'split_val': 242}


In [3]:
# Fig 1 — phân bố độ dài nhãn
ax = df['n_chars'].plot(kind='hist', bins=range(1, 27), edgecolor='black')
ax.set_title('Phân bố độ dài nhãn (số ký tự)')
ax.set_xlabel('n_chars'); ax.set_ylabel('số mẫu')
plt.tight_layout(); plt.savefig(FIG / 'fig_len_hist.png', dpi=110)
print('fig_len_hist.png')

fig_len_hist.png


In [4]:
# Fig 2 — loại hình mark (mark drawing)
vc = df['drawing_kind'].value_counts()
ax = vc.plot(kind='barh')
ax.set_title('Số mẫu theo loại hình mark'); ax.set_xlabel('số mẫu')
plt.tight_layout(); plt.savefig(FIG / 'fig_drawing_kind.png', dpi=110)
print(vc.to_dict())

{'standard_character': 3341, 'design_with_words': 853, 'stylized_words': 410, 'typeset_old': 236}


In [5]:
# Fig 3 — số từ / nhãn
nwords = df['word_mark'].str.split().map(len)
ax = nwords.value_counts().sort_index().plot(kind='bar')
ax.set_title('Phân bố số từ mỗi nhãn'); ax.set_xlabel('số từ'); ax.set_ylabel('số mẫu')
plt.tight_layout(); plt.savefig(FIG / 'fig_nwords.png', dpi=110)
print('mean words/label =', round(float(nwords.mean()), 3))

mean words/label = 1.692


In [6]:
# Fig 4 — kích thước ảnh (mẫu tối đa 500 ảnh)
sample = df.sample(min(500, len(df)), random_state=1111)
sizes = []
for rel in sample['image_path']:
    p = DATA_ROOT / rel
    if p.exists():
        with Image.open(p) as im:
            sizes.append(im.size)
w = np.array([s[0] for s in sizes]); h = np.array([s[1] for s in sizes])
ax = plt.figure(figsize=(6, 4)).gca()
ax.hist(w, bins=30, alpha=0.6, label='width')
ax.hist(h, bins=30, alpha=0.6, label='height')
ax.set_title(f'Kích thước ảnh (n={len(sizes)})'); ax.set_xlabel('px'); ax.legend()
plt.tight_layout(); plt.savefig(FIG / 'fig_img_size.png', dpi=110)
print('median wxh =', int(np.median(w)), 'x', int(np.median(h)))

median wxh = 600 x 250


In [7]:
# Fig 5 — tần suất ký tự (top 30) + phủ chữ Việt
from collections import Counter
freq = Counter(ch for s in df['word_mark'] for ch in s)
top = freq.most_common(30)
ax = pd.Series(dict(top)).plot(kind='bar')
ax.set_title('Top 30 ký tự xuất hiện trong nhãn')
plt.tight_layout(); plt.savefig(FIG / 'fig_char_freq.png', dpi=110)
vn = 'ăâđêôơưĂÂĐÊÔƠƯáàảãạếệ'
n_vn = sum(1 for s in df['word_mark'] for ch in s if ch in vn)
print('top10:', top[:10])
print('số ký tự tiếng Việt có dấu trong nhãn US:', n_vn)

top10: [('E', 5116), ('A', 4299), ('O', 3520), ('I', 3460), (' ', 3348), ('R', 3293), ('T', 3013), ('N', 2965), ('S', 2875), ('L', 2595)]
số ký tự tiếng Việt có dấu trong nhãn US: 1


## Nhận xét
- Phần lớn nhãn US là **standard character** (chữ đánh máy) + một phần chữ cách điệu.
- Độ dài nhãn dồn về 6–14 ký tự (khớp `max_label_length=25`).
- Ảnh kích thước đa dạng, thường nhỏ/rộng → `img_size=[32,128]` với resize bicubic.
- Có xuất hiện ký tự có dấu (dấu hiệu thương hiệu nước ngoài) → ủng hộ charset tiếng Việt.

**Lưu ý**: notebook không xác nhận *bằng mắt* nhãn↔ảnh (môi trường này không đọc ảnh).
Kiểm tra khớp nhãn/ảnh nên làm thủ công vài mẫu từ `fig_*` + `raw/images_us/`.
Phân tích lỗi sâu hơn ở `notebooks/02_error_analysis.ipynb` (nhánh 13).